# Office hours: loading your own CSV or Google Sheet, labeling it with Gemini, and running `evaluate()`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/team-simmer/ai4tm/blob/main/week_3/office_hours_load_csv_and_evaluate.ipynb)

The [evaluation template](lesson12_evaluation_template.ipynb) ends with a reusable `evaluate()` function and a three-line note on using it with your own data. This walkthrough fills in the gap between that note and a working result, using a Google Search Console export as the example: getting your data out of a file on your computer or a Google Sheet and into a pandas table, having Gemini assign a topic to every row, cleaning the labels up enough that they actually match, and running the lot through `evaluate()`.

**Time**: ~25 minutes
**Cost**: A fraction of a cent. Step 3 makes one short Gemini call per row, and estimates the cost before it runs.

> **Running this locally, or in VS Code instead of Colab?** See [Week 2's setup guide](../week_2/lesson08_setup_guide.ipynb). This notebook uses the same `week_3` environment as the evaluation template.

## What your data needs to look like

One row per item, with two columns (the model's answers are added by Step 3):

| Column | What goes in it |
|---|---|
| the text | what's being labeled: the search query, the feedback, the ad copy |
| the correct answer | the label a person assigned (the *ground truth*) |

Column names don't matter, you'll tell the notebook which is which in a moment.

**Privacy reminder from Week 4**: only use public, synthetic, or anonymized data here. If your sheet contains customer names, emails, or anything else personal, scrub it first (see the [PII protection notebook](../week_4/lesson17_pii_protection.ipynb)).

## Setup

**In Colab**: the cell below installs everything needed. Step 3 reads your Gemini API key from Colab Secrets, saved as `GEMINI_API_KEY`, the same way as the lesson notebooks. If you haven't done that yet, follow [Week 2's setup guide](../week_2/lesson08_setup_guide.ipynb) first.

**Locally**: use the same `week_3/.venv` environment you set up for the evaluation template, then run the cell below.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab already ships most of these, this just makes sure nothing is missing.
    %pip install -q google-genai pandas scikit-learn matplotlib seaborn gspread
else:
    # Local: installs from week_3/requirements.txt, looked up by directory so the
    # repo's top-level requirements.txt isn't picked up by mistake.
    import pathlib
    _cwd = pathlib.Path.cwd()
    if _cwd.name == "week_3" and (_cwd / "requirements.txt").exists():
        _req_path = "requirements.txt"
    elif (_cwd / "week_3" / "requirements.txt").exists():
        _req_path = "week_3/requirements.txt"
    elif (_cwd.parent / "week_3" / "requirements.txt").exists():
        _req_path = "../week_3/requirements.txt"
    else:
        _req_path = None

    if _req_path is None:
        print("Couldn't find week_3/requirements.txt from the current working directory:", _cwd)
        print("Set up your local environment first -- see Week 2's setup guide for the uv commands.")
    else:
        %pip install -q -r {_req_path}

## Step 0: a sample file to practice on

The sample is a Google Search Console export for Larkspur Trail Co., the invented outdoor brand you'll meet again in Week 7 (same queries, same numbers). It's laid out the way Search Console actually exports the **Queries** tab: a `Top queries` column, clicks, impressions, CTR as a percentage, and average position, sorted by clicks.

Search Console doesn't know what your queries are *about*, so one column has been added on the right, the way you would in the sheet yourself: `Topic`, the topic a person assigned to each query. That's the ground truth. The model's topics don't exist yet, Step 3 has Gemini fill them in.

The eight topics roughly follow the site's sections: shipping, returns, sizing, rain jackets, layering, daypacks, account, and the company itself. Like a sheet filled in by hand, it has a few labels with different capitalisation or a stray space, and an empty row at the bottom. That's deliberate, it's what the cleanup step later is for.

Run the cell below to write it as a CSV next to this notebook.

In [ ]:
SAMPLE_CSV = """Top queries,Clicks,Impressions,CTR,Position,Topic
larkspur login,88,900,9.78%,1.1,account & login
larkspur return policy,70,1100,6.36%,2.1,Returns
how long does shipping take,60,1400,4.29%,4.2,Shipping & Delivery
larkspur size guide,55,700,7.86%,1.8,Sizing & Fit
how do i return an item,35,640,5.47%,4.4,Returns
how to layer for cold weather hikes,34,410,8.29%,2.9,Layering
when will my order arrive,30,900,3.33%,6.5,Shipping & Delivery
merino wool vs synthetic base layer,26,340,7.65%,4.0,Layering
larkspur trailhead daypack,24,260,9.23%,2.0,Daypacks
how do i know my jacket size,22,320,6.88%,3.3,Sizing & Fit
30 day return policy,20,300,6.67%,3.9,Returns
size chart,19,260,7.31%,3.0,Sizing & Fit 
free returns,15,260,5.77%,5.0,Returns
best base layer for cold weather running,15,190,7.89%,5.5,Layering
larkspur shipping time,14,380,3.68%,5.1,Shipping & Delivery
sign in to my larkspur account,14,150,9.33%,1.9,Account & Login
best jacket for hiking in the rain,12,610,1.97%,12.5,Rain Jackets
are larkspur jackets true to size,11,180,6.11%,4.9,Sizing & Fit
what to wear under a rain shell in winter,10,130,7.69%,5.8,Layering
is larkspur trail co sustainable,9,140,6.43%,3.1,Company & Brand
rain shell vs windbreaker,9,520,1.73%,14.0,Rain Jackets
shipping estimate,9,250,3.60%,7.0,Shipping & Delivery
larkspur trail co account,8,95,8.42%,2.4,Account & Login
how to choose a waterproof jacket,8,480,1.67%,15.2,Rain Jackets
how many days for delivery,8,210,3.81%,6.8,Shipping & Delivery
larkspur trail co shipping cost,6,170,3.53%,8.3,Shipping & Delivery
larkspur sustainability,5,70,7.14%,2.6,Company & Brand
waterproof vs water resistant jacket,5,260,1.92%,13.8,Rain Jackets
22l daypack larkspur,5,60,8.33%,3.5,Daypacks
can i return worn hiking boots,4,90,4.44%,8.2,Returns
do i need a rain shell for day hikes,4,190,2.11%,16.4,Rain Jackets
3 layer vs 2.5 layer rain jacket,3,210,1.43%,18.0,Rain Jackets
what to look for in a rain jacket for hiking,2,150,1.33%,19.5,Rain Jackets
trail co coupon code reddit,1,12,8.33%,30.0,Company & Brand
larkspur recall 2024,0,5,0.00%,55.0,Company & Brand
larkspur trail co warehouse jobs,0,6,0.00%,48.0,Company & Brand
larkspur trail co ceo,0,8,0.00%,40.0,Company & Brand
larkspur trail co internship,0,4,0.00%,62.0,Company & Brand
,,,,,
"""

SAMPLE_PATH = "office_hours_search_console_topics.csv"
with open(SAMPLE_PATH, "w", encoding="utf-8") as f:
    f.write(SAMPLE_CSV)
print(f"Wrote {SAMPLE_PATH}")

### Doing this with your own Search Console data

1. In Search Console open **Performance → Search results**, set the date range, and switch to the **Queries** tab below the chart.
2. Click **Export** (top right) and pick **Google Sheets**, or **Download CSV** (you get a zip; the file you want inside it is `Queries.csv`).
3. Add one column with your own topic for each query. Step 3 of this notebook adds the model's topics.
4. Load the sheet or CSV with one of the options below.

Search queries are aggregated and anonymized by Google, but check before sharing a sheet by link: people sometimes type names, emails or order numbers into a search box.

## Step 1: load your data

Pick the option that matches where your data is. Each one loads the data into a table called `raw` and prints its column names. Options B and C do nothing until you paste a Sheet URL into them, so running every cell top to bottom is safe: with no URL set, the data comes from Option A.

- **Option A**: a CSV file, on your computer (running locally) or uploaded to Colab. This is also how the sample file above loads.
- **Option B**: a Google Sheet shared as "Anyone with the link can view". Works in Colab and locally.
- **Option C**: a Google Sheet that has to stay private. Colab only.

In Colab, click the arrow next to an option's heading to fold it away and keep only the one you're using on screen.

After loading, check the column names in the output carefully: you'll copy them into Step 2, and a trailing space in a header is invisible in a spreadsheet but very visible to Python.

### Option A: a CSV file

This is just `pd.read_csv(CSV_PATH)`. Where the path points depends on where the notebook runs:

- **Locally**: any path on your computer, e.g. `/Users/you/Downloads/Queries.csv`. Relative paths are relative to this notebook's folder, so a CSV sitting next to the notebook only needs its file name.
- **In Colab**: Colab runs on a Google server that can't see your disk, so a path like `/Users/...` or `C:\Users\...` won't work there. Put the file onto the Colab machine first: click the **folder icon** in the left sidebar, drag your CSV into the panel, then right-click it → **Copy path** and paste that into `CSV_PATH`. It'll look like `/content/Queries.csv`. Uploaded files are deleted when the runtime disconnects, so you'll need to upload it again next session.

Leave `CSV_PATH` as it is to load the sample file from Step 0.

In [ ]:
import pandas as pd

CSV_PATH = "office_hours_search_console_topics.csv"

raw = pd.read_csv(CSV_PATH)
print(f"{len(raw)} rows, columns: {list(raw.columns)}")
raw.head()

### Option B: a Google Sheet shared by link

The simplest Sheet route, works both in Colab and locally, no login from code.

1. In the Sheet: **Share** → General access → **Anyone with the link** → Viewer.
2. Copy the URL from the address bar into `GOOGLE_SHEET_URL` below. If the sheet has several tabs, open the tab you want first, so the URL ends in `#gid=...`.

The cell does nothing while `GOOGLE_SHEET_URL` is empty, so running every cell in order is safe. The comment above it has a link to the sample data, shared by link, to try this option with.

Every Google Sheet can be downloaded as CSV by swapping the end of its URL for `/export?format=csv`. The function below builds that URL from whatever you paste and hands it straight to `pd.read_csv`, which reads URLs the same way it reads files.

Anyone with the link can see the sheet, so this is only for synthetic or already-anonymized data. If you get an HTML error page or an `HTTP Error 401`, the sheet isn't shared by link yet. `HTTP Error 400` usually means the `#gid=` in the URL points at a tab that no longer exists.

In [ ]:
import re
import pandas as pd

# Leave empty to skip this option. To try it with the sample data, paste:
# https://docs.google.com/spreadsheets/d/1ZZfTnhocpWCeB_XuJ6cueK0N6sFJ_VSn_WZoNCu3N6s/edit?usp=sharing
GOOGLE_SHEET_URL = ""

def google_sheet_csv_url(sheet_url):
    match = re.search(r"/spreadsheets/d/([a-zA-Z0-9-_]+)", sheet_url)
    if not match:
        raise ValueError("That doesn't look like a Google Sheets URL (expected .../spreadsheets/d/<id>/...).")
    sheet_id = match.group(1)
    csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
    # Without a gid Google exports the first tab. Don't guess gid=0 for it: a first tab
    # that was added later has a different id, and a wrong gid gives "HTTP Error 400".
    gid_match = re.search(r"[#&?]gid=(\d+)", sheet_url)
    return csv_url + f"&gid={gid_match.group(1)}" if gid_match else csv_url

if GOOGLE_SHEET_URL:
    raw = pd.read_csv(google_sheet_csv_url(GOOGLE_SHEET_URL))
    print(f"{len(raw)} rows, columns: {list(raw.columns)}")
    display(raw.head())
else:
    print("Option B skipped: no GOOGLE_SHEET_URL set.")

### Option C: a private Google Sheet, in Colab

For a sheet you don't want to share by link. Colab asks you to sign in with your Google account and allow access, then reads the sheet as you, so it works for anything you can open yourself. `gspread` is the library that talks to Sheets.

This only works in Colab: locally you'd need to set up a Google Cloud service account, which is more setup than an office-hours walkthrough is worth. If you're local and the sheet must stay private, use **File → Download → Comma-separated values (.csv)** in Sheets and then use Option A.

Paste the Sheet's URL into `GOOGLE_SHEET_URL` below (with the right tab open); while it's empty the cell does nothing. Note that `get_all_records()` returns numbers as numbers and the header row as column names, the same result you'd get from the CSV routes.

In [ ]:
import re
import pandas as pd

GOOGLE_SHEET_URL = ""     # leave empty to skip this option

if GOOGLE_SHEET_URL:
    # gspread and google.colab come with Colab; this option doesn't work locally
    import gspread
    from google.auth import default
    from google.colab import auth

    auth.authenticate_user()           # pops up a Google sign-in and permission prompt
    creds, _ = default()
    gc = gspread.authorize(creds)

    spreadsheet = gc.open_by_url(GOOGLE_SHEET_URL)
    gid_match = re.search(r"[#&?]gid=(\d+)", GOOGLE_SHEET_URL)
    worksheet = spreadsheet.get_worksheet_by_id(int(gid_match.group(1))) if gid_match else spreadsheet.sheet1
    print(f"Reading '{spreadsheet.title}' / tab '{worksheet.title}'")

    raw = pd.DataFrame(worksheet.get_all_records())
    print(f"{len(raw)} rows, columns: {list(raw.columns)}")
    display(raw.head())
else:
    print("Option C skipped: no GOOGLE_SHEET_URL set.")

## Step 2: tell the notebook which column is which

Copy the column names from the output above. `PRED_COLUMN` is the name of the new column Step 3 writes the model's topics into.

`CATEGORIES` is your list of allowed labels, spelled exactly how you want them to appear in the report. It's also used to fix up labels that differ only in capitalisation or spacing, so leave it as `None` only if you have no fixed list.

In [ ]:
TEXT_COLUMN = "Top queries"
TRUE_COLUMN = "Topic"
PRED_COLUMN = "Model topic"       # new column, filled in by Step 3

CATEGORIES = [
    "Shipping & Delivery",
    "Returns",
    "Sizing & Fit",
    "Rain Jackets",
    "Layering",
    "Daypacks",
    "Account & Login",
    "Company & Brand",
]

## Step 3: fill in the model's topics with Gemini

Your data has your topics; now it needs the model's. The cell below sends each query to Gemini on its own, with the list of allowed topics, and asks for exactly one back. Gemini never sees your `Topic` column, the same rule as the evaluation template: a model that can see the answer isn't being tested.

This is `classify()` from the [evaluation template](lesson12_evaluation_template.ipynb) with the prompt changed from customer feedback to search queries. Two parts of it are worth adapting to your own data:

- `BUSINESS_CONTEXT`: one line on what the site is. "free returns" means something different on an outdoor-gear shop than on a tax-software site, and the model can't know which unless you tell it.
- `CATEGORIES` from Step 2: the model can only pick from that list, and anything it answers that isn't on it exactly is recorded as `UNKNOWN`, rather than silently dropped.

**Estimate the cost first**, the habit from [Week 2's token cost guide](../week_2/lesson09_token_cost_guide.ipynb). The cell below does it without calling the API.

In [ ]:
MODEL_NAME = "gemini-3.5-flash-lite"
BUSINESS_CONTEXT = "Larkspur Trail Co., an online shop for outdoor gear: rain jackets, base layers and daypacks"

queries = raw[TEXT_COLUMN].dropna().astype(str)

# Same rough estimate as Week 2: 1 token is about 4 characters. Each call sends the prompt
# (~500 characters with the topic list) plus the query, and gets back a topic name (~5 tokens).
# Prices are USD per 1M tokens for Gemini 3.5 Flash-Lite -- check current pricing, it changes.
PRICE_IN, PRICE_OUT = 0.30, 2.50
input_tokens = sum((500 + len(q)) / 4 for q in queries)
output_tokens = len(queries) * 5
cost = input_tokens / 1_000_000 * PRICE_IN + output_tokens / 1_000_000 * PRICE_OUT

print(f"{len(queries)} queries to classify with {MODEL_NAME}, estimated cost: ${cost:.4f}")

In [ ]:
import time
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

TOPIC_LIST = "\n".join(f"- {c}" for c in CATEGORIES)

def classify_query(query, retries=2):
    prompt = f"""This is a search query that brought people from Google to the website of {BUSINESS_CONTEXT}.
Assign it to exactly ONE of these topics:
{TOPIC_LIST}

Query: "{query}"

Respond with only the topic name, exactly as written above. Nothing else."""

    for attempt in range(retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                # temperature=0 keeps the answers as repeatable as possible between runs
                config=types.GenerateContentConfig(temperature=0),
            )
            label = response.text.strip()
            return label if label in CATEGORIES else "UNKNOWN"
        except Exception as e:
            if attempt == retries:
                print(f"  Giving up on {query!r} after {retries} retries ({e}), recording as UNKNOWN.")
                return "UNKNOWN"
            time.sleep(2)

predictions = []
for i, query in enumerate(raw[TEXT_COLUMN]):
    if pd.isna(query) or not str(query).strip():
        predictions.append(None)   # empty row, nothing to classify -- Step 4 drops it
    else:
        predictions.append(classify_query(str(query)))
    if (i + 1) % 10 == 0:
        print(f"  {i + 1} of {len(raw)} done")
raw[PRED_COLUMN] = predictions

unknown_count = (raw[PRED_COLUMN] == "UNKNOWN").sum()
if unknown_count:
    print(f"⚠️ {unknown_count} answer(s) didn't match a topic exactly (or the call failed), recorded as UNKNOWN.")
print("Done.")

raw[[TEXT_COLUMN, TRUE_COLUMN, PRED_COLUMN]].head(10)

## Step 4: clean the labels

This is where most "why is my accuracy so low?" questions end. To Python, `"Sizing & Fit"`, `"sizing & fit"` and `"Sizing & Fit "` are three different categories, so a prediction that's actually right gets scored as wrong, and the confusion matrix sprouts extra rows. The function below:

1. Renames your columns to the standard `text` / `true_label` / `predicted_label` (every other column, like clicks and impressions, is kept as it is).
2. Drops rows where either label is empty (blank lines at the end of a sheet are the usual culprit).
3. Trims stray spaces and matches labels to `CATEGORIES` ignoring case.
4. Prints every change it made, so nothing is fixed silently.

Anything still not in `CATEGORIES` afterwards is left as-is and flagged: that's either a typo to fix in the source, or a genuinely new category you need to add to the list.

In [ ]:
def clean_labels(raw, true_col, pred_col, text_col=None, categories=None):
    missing = [c for c in (true_col, pred_col, text_col) if c is not None and c not in raw.columns]
    if missing:
        raise KeyError(f"Column(s) {missing} not found. Available columns: {list(raw.columns)}")

    columns = {true_col: "true_label", pred_col: "predicted_label"}
    if text_col is not None:
        columns[text_col] = "text"
    df = raw.rename(columns=columns)

    # 1. Drop rows with a missing label
    before = len(df)
    df = df.dropna(subset=["true_label", "predicted_label"])
    for col in ("true_label", "predicted_label"):
        df[col] = df[col].astype(str).str.strip()
    df = df[(df["true_label"] != "") & (df["predicted_label"] != "")].reset_index(drop=True)
    if len(df) < before:
        print(f"Dropped {before - len(df)} row(s) with an empty label.")

    # 2. Match labels to the canonical spelling, ignoring case and extra spaces
    if categories is not None:
        canonical = {" ".join(c.lower().split()): c for c in categories}
        for col in ("true_label", "predicted_label"):
            original = df[col]
            df[col] = original.map(lambda v: canonical.get(" ".join(v.lower().split()), v))
            changed = (original != df[col]).sum()
            if changed:
                print(f"Normalised {changed} value(s) in {col}.")

        unknown = sorted((set(df["true_label"]) | set(df["predicted_label"])) - set(categories))
        if unknown:
            print(f"⚠️ Labels not in CATEGORIES (typo, or a category to add?): {unknown}")

    print(f"\n{len(df)} rows ready to evaluate.")
    return df

df = clean_labels(raw, TRUE_COLUMN, PRED_COLUMN, TEXT_COLUMN, CATEGORIES)
df.head()

In the sample there's an empty row at the bottom, one topic typed in lower case (`account & login`), and one with a trailing space (`Sizing & Fit `, trimmed before matching, so it isn't in the "Normalised" count). Without this step those rows would be scored as wrong whatever the model said, and the report would list them as extra categories.

Gemini's own answers don't need this: Step 3 already records anything that isn't an exact topic name as `UNKNOWN`. If any show up, they're flagged here and `evaluate()` keeps `UNKNOWN` as its own row and column, so they can't disappear from the count.

## Step 5: run `evaluate()`

This is the same function as the end of the [evaluation template](lesson12_evaluation_template.ipynb), copied in unchanged so this notebook runs on its own. Every Colab notebook is a separate session, so a function defined in another notebook isn't available here until you run its definition again.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate(y_true, y_pred, labels=None, title="Evaluation Results"):
    """
    Evaluate classification predictions against ground truth.
    Works on ANY labeled classification task -- not just this dataset.

    y_true, y_pred : lists or pandas Series of labels (same length, same order)
    labels         : ordered list of all possible category names (optional --
                      inferred from the data if not given)
    """
    observed = set(y_true) | set(y_pred)
    if labels is None:
        labels = sorted(observed)
    else:
        # Keep any label the caller passed in, but don't silently drop a label that shows
        # up in the data and wasn't listed (e.g. an UNKNOWN/fallback value) -- dropping it
        # would make accuracy and the confusion matrix disagree on how many rows there are.
        labels = list(labels) + sorted(observed - set(labels))

    accuracy = accuracy_score(y_true, y_pred)
    print(f"{title}")
    print("=" * len(title))
    print(f"Overall accuracy: {accuracy:.1%}\n")

    print("Per-category breakdown (precision / recall / F1):")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return {"accuracy": accuracy, "confusion_matrix": cm}

In [ ]:
results = evaluate(df["true_label"], df["predicted_label"], labels=CATEGORIES, title="My Evaluation")

## Step 6: read the rows it got wrong

Same habit as the evaluation template: the numbers say how much disagreement there is, only the rows say whether it's the model, the label, or two topics that overlap. In the sample, look out for queries that sit between two topics, like "what to wear under a rain shell in winter" (layering, or rain jackets?) or "free returns" (returns, or shipping?). Whether Gemini lands on the same side as the `Topic` column can change from run to run and model to model.

Search Console data adds one thing the evaluation template didn't have: not every row is equally important. Accuracy counts a query with 12 impressions the same as one with 1,400. If the topics feed a content report or a budget decision, a mistake on a high-traffic query matters much more, so the mismatches below are sorted by impressions, and the cell also reports what share of total impressions landed in the wrong topic.

Compare the two numbers: if the share of impressions is much lower than the share of wrong rows, the mistakes are mostly on small long-tail queries; if it's higher, they're on the queries that matter most. Neither number is the "real" one, which one matters depends on what the topics will be used for.

In [ ]:
WEIGHT_COLUMN = "Impressions"      # or "Clicks", or None if your data has no such column

mismatches = df[df["true_label"] != df["predicted_label"]]
print(f"{len(mismatches)} of {len(df)} predictions disagreed with the label.")

if WEIGHT_COLUMN in df.columns:
    # Exports and sheets sometimes format numbers as text ("1,400"), so convert first
    weight = pd.to_numeric(df[WEIGHT_COLUMN].astype(str).str.replace(",", ""), errors="coerce").fillna(0)
    wrong_share = weight[mismatches.index].sum() / weight.sum()
    print(f"Those rows carry {wrong_share:.1%} of all {WEIGHT_COLUMN.lower()}.")
    mismatches = mismatches.assign(_weight=weight[mismatches.index]).sort_values("_weight", ascending=False)
print()

for _, row in mismatches.iterrows():
    if "text" in df.columns:
        print(f'"{row["text"]}"' + (f'  ({int(row["_weight"]):,} {WEIGHT_COLUMN.lower()})' if "_weight" in row else ""))
    print(f'  labeled:    {row["true_label"]}')
    print(f'  predicted:  {row["predicted_label"]}')
    print()

## Step 7 (optional): save the results

Writes the cleaned rows, including Gemini's topics, plus a `correct` column: handy for going through the mismatches with whoever labeled the data, or for pasting the model's topics back into your Sheet. In Colab the second part downloads the file to your computer; locally it's saved next to this notebook.

In [ ]:
OUTPUT_PATH = "office_hours_evaluated.csv"

out = df.assign(correct=df["true_label"] == df["predicted_label"])
out.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH}")

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_PATH)

## Common errors

| Error | Usual cause | Fix |
|---|---|---|
| `FileNotFoundError` | Path is relative to a different folder, or you're in Colab pointing at your own disk | Use an absolute path locally; in Colab, upload the file and use **Copy path** (Option A) |
| `UnicodeDecodeError` | Excel saved the file in a non-UTF-8 encoding | Re-save it from Excel as "CSV UTF-8", or add `encoding="cp1252"` to `pd.read_csv` |
| One column containing `a;b;c` | Semicolon-separated file (Excel in many European locales) | Add `sep=";"` to `pd.read_csv` |
| `HTTP Error 401` / HTML instead of data | Google Sheet not shared by link | Share → Anyone with the link → Viewer, or use Option C |
| `KeyError` in Step 4 | Column name doesn't match exactly | Copy the names from the list printed after loading |
| Accuracy far lower than expected | Label spellings differ between columns | Check the "Normalised" and "⚠️ Labels not in CATEGORIES" lines from Step 4 |
| Extra categories in the confusion matrix | Same as above | Fix the source, or add the category to `CATEGORIES` |
| `SecretNotFoundError` in Step 3 | Key not saved in Colab Secrets, or the notebook access toggle is off | Follow Week 2's setup guide |
| Lots of `UNKNOWN` in Step 3 | The model is answering with names that don't exactly match `CATEGORIES` | Print a few `response.text` values; tighten the prompt or simplify the topic names |

**Trying a different model or prompt?** Change `MODEL_NAME`, `BUSINESS_CONTEXT` or the prompt in Step 3, then rerun Step 3 onwards: it overwrites the previous `Model topic` column. Running `evaluate()` on each version is the fair way to compare them: same queries, same answer key.